# Feature Selection & Audit
## Project: Amazon Employee Access Challenge

This notebook analyzes categorical features and their combinations (Feature Engineering).
We are looking for combinations that possess high mutual information but do not lead to overfitting due to high cardinality.

In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import sys
sys.path.append('../')
from src.feature_audit import screen_feature_pairs


df = pd.read_csv('../data/train.csv')
y = df['ACTION']
X =  df.drop(columns=['ACTION', 'ROLE_CODE', 'ROLE_FAMILY'], axis=1)
cat_cols = X.columns.tolist()

print(f"Dataset size: {X.shape}")

Dataset size: (32769, 9)


### 1. Feature Pair Screening

We will iterate through all possible combinations of the base columns (28 pairs). For each pair, we will calculate:
* **Cardinality Ratio:** the proportion of unique values, where higher indicates a greater risk of overfitting.
* **Singleton Ratio:** the proportion of categories that appear exactly once.
* **Rare Rows Coverage:** the percentage of rows in the entire dataset that will be lost or collapsed into the OTHER bin when applying a frequency filter.
* **Target Std:** the standard deviation of the target variable across categories (the higher the spread of approval/rejection probabilities between groups, the more useful the feature is for splitting).
* **Mutual Information (MI):** the strength of the statistical relationship with the target variable.


In [2]:
leaderboard = screen_feature_pairs(X, y, cat_cols, top_n=10, min_freq=3)
leaderboard

,pair,cardinality_ratio,singleton_ratio,rare_rows_coverage,target_std,mutual_info
0,RESOURCE__MGR_ID,0.843053,0.853218,0.913699,0.097073,0.213900
1,RESOURCE__ROLE_DEPTNAME,0.529586,0.670566,0.525405,0.103812,0.179688
2,RESOURCE__ROLE_FAMILY_DESC,0.693765,0.829462,0.715280,0.141122,0.178606
3,RESOURCE__ROLE_TITLE,0.581129,0.754871,0.593030,0.133623,0.162876
4,RESOURCE__ROLE_CODE,0.581129,0.754871,0.593030,0.133623,0.162876
5,RESOURCE__ROLE_ROLLUP_2,0.409594,0.624497,0.393848,0.129714,0.153020
6,MGR_ID__ROLE_FAMILY_DESC,0.243309,0.330365,0.175105,0.161220,0.139003
7,MGR_ID__ROLE_TITLE,0.228387,0.289818,0.150905,0.161610,0.136285
8,MGR_ID__ROLE_CODE,0.228387,0.289818,0.150905,0.161610,0.136285
9,RESOURCE__ROLE_FAMILY,0.402667,0.642440,0.390949,0.129610,0.132576


###  2. Result Analysis and Feature Selection
Based on the resulting table, I decided to use the following pairs for the XGBoost model:

✅ RESOURCE_ROLE_DEPTNAME
Shows one of the highest Mutual Information scores. The cardinality remains within reasonable limits since the resource is logically tied to a specific department. This pair provides a stable boost during cross-validation.

✅ ROLE_ROLLUP_1, ROLE_ROLLUP_2, ROLE_DEPTNAME, ROLE_TITLE
Hierarchical features with high density and minimal noise (low Singleton Ratio).

❌ Why I am excluding RESOURCE_MGR_ID:
Although the base combination of Resource and Manager shows an abnormally high Mutual Information, its Cardinality Ratio exceeds 80%. This means most of these pairs are unique to each row. The high MI here is an illusion (memorizing the dataset), which will inevitably lead to XGBoost overfitting.

### 3. Transformation (Pipeline)
For the final training pipeline, I will apply rare value filtering (collapsing them into OTHER) to protect the XGBoost trees from the noise
of singleton categories. Also I am adding frequency encoding for the key columns (RESOURCE, MGR_ID, ROLE_DEPTNAME) so XGBoost can also rely on whether an instance is unique or occurs frequently in the dataset.